# Assignment: Real-World Hand Pose Estimation (YOLOv26-Pose)
### Master Laboratory Guide - Moving Beyond Toy Data

---

## Project Overview & Objectives
Until today, you have trained lightweight toy architectures on flat, non-complex datasets like MNIST or Fashion-MNIST. Those jobs ran to completion in minutes, insulating you from the engineering realities of training deep neural networks.

In this assignment, you will deploy, optimize, and serve a state-of-the-art **YOLOv26-Pose** model. This framework leverages an **End-to-End One-to-One Head** to perform **Direct Set Prediction**, completely bypassing traditional, non-differentiable Post-Processing steps like Non-Maximum Suppression (NMS). Your model will be trained on a complex coordinate matrix to map and track **21 distinct structural hand joints (keypoints)** simultaneously.

### CRITICAL WARNING: THE TIME COST OF DEEP LEARNING
This is a computationally intensive training session. Depending on your target hardware configuration, completing your training schedule will require between **1 to 15 hours of continuous execution, or even 5 sessions of 3 hour training on google colab**.
* **If you start this assignment the night before the deadline, it is mathematically impossible for you to complete it.**
* Free cloud platforms enforce aggressive idle-timeout policies. If you leave your session unattended, your progress will be wiped. You must proactively monitor, log, checkpoint, and utilize the environments correctly.

---

## SUBMISSION REQUIREMENTS
You will **NOT** upload any raw files, code, or video clips directly to Moodle. Instead, this assignment evaluates your ability to build a professional, public engineering portfolio. **You must submit a single link to a public GitHub repository** containing your structured source code, environment configurations, automated video recordings, and a professional markdown report.
## Two-Phase Submission Structure & Milestones

To prevent catastrophic late-stage execution failures due to the hours-long training budget of this project, the submission is strictly split into two mandatory phases. 

---

### Phase 1: Infrastructure Validation & Baseline Record
* **Deadline:** Tuesday, June 30, 2026, at 23:59
* **Submission Type:** Public GitHub Repository Link (Submitted via Moodle)

You must read all environmental instructions, provision your isolated dependencies using `uv`, register an official account on GitHub, and establish your workspace. You must execute `detect.py` using the base pretrained weights to record your initial performance boundary.

#### Mandatory Repository Checklist for Phase 1:
* Fully configured `pyproject.toml` file matching course dependencies.
* An active, configured `.gitignore` file isolating your private keys and virtual environments.
* The baseline recording artifact saved precisely under: `runs/pose/predict/failed_attempt.mp4`.

---

### Phase 2: Complete Optimization Loop & Analytical Reporting
* **Deadline:** Thursday, July 9, 2026, at 23:59 (1.5 weeks following Phase 1)
* **Submission Type:** Updated GitHub Repository Link (Re-evaluation via Moodle)

During this phase, you will execute your deep training cycle (either through local compute blocks or timeout-resilient cloud intervals). You will tune hyperparameters, track real-time gradient behaviors, implement your fine-tuned weights locally, and finalize your engineering report.

#### Mandatory Repository Checklist for Phase 2:
* Complete implementation code inside `hand_pose_lab.ipynb`.
* The optimized fine-tuned tracking recording saved under: `runs/pose/predict/successful_attempt.mp4`.
* A finished, structured `README.md` file populating the complete Analytical Report Blueprint (including your synchronized Weights & Biases loss curves and convergence analysis).

---

### Late Submission Policy
* **Missing Phase 1 automatically caps your overall laboratory score at 50%**, even if your final model tracks flawlessly. 
* Procrastinating will result in cloud timeout disconnections or local VRAM out-of-memory errors that cannot be resolved an hour before the final deadline. Verify your execution layers early.

## Step 1: Local Environment Provisioning

This course strictly uses **uv** for fast, reliable Python package and virtual environment isolation. Do not use standard `pip` or `conda` commands globally.

### Your Local Configuration Files
To enable simple environment locking, ensure you have a `pyproject.toml` file in your root folder containing the following configuration:

```toml
[project]
name = "hand-pose-estimation"
version = "0.1.0"
description = "YOLOv26 Hand Keypoint Detection and Tracking Lab"
readme = "README.md"
requires-python = ">=3.10"
dependencies = [
    "torch>=2.0.0",
    "torchvision>=0.15.0",
    "ultralytics>=8.0.0",
    "wandb>=0.16.0",
    "python-dotenv>=1.0.0",
    "opencv-python>=4.8.0",
]

[tool.uv]
package = false
```

Run `uv sync` in your system terminal to lock the environment before proceeding inside this notebook.

In [1]:
import os
import sys
import time
import cv2
import torch
from ultralytics import YOLO
import wandb
from dotenv import load_dotenv

# Enforce secure and explicit hardware device allocation
device = ('cuda' if torch.cuda.is_available() 
          else 'mps' if torch.backends.mps.is_available() 
          else 'cpu')
print(f"CRITICAL RESOURCE ALLOCATION -> Active Hardware Device: {device.upper()}")

CRITICAL RESOURCE ALLOCATION -> Active Hardware Device: MPS


In [2]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
ROOT_OUTPUT_DIR = PROJECT_DIR / "runs" / "pose"

print("CURRENT WORKING DIRECTORY:", PROJECT_DIR)
print("ROOT_OUTPUT_DIR:", ROOT_OUTPUT_DIR)

CURRENT WORKING DIRECTORY: /Users/itzikdermer/Documents/Python project/Idan Tobis - Deep Learning/HW /Project 3
ROOT_OUTPUT_DIR: /Users/itzikdermer/Documents/Python project/Idan Tobis - Deep Learning/HW /Project 3/runs/pose


## Step 2: Credentials and Infrastructure Security

Hardcoding private API authorization keys inside shared files is a severe compliance violation. We enforce local environment isolation using `python-dotenv`.

### Configuration Instructions:
1. Register at [Weights & Biases](https://wandb.ai/site) using your **official university email address** to unlock premium academic tier benefits.
2. Copy your private authorization token from [https://wandb.ai/authorize](https://wandb.ai/authorize).
3. Create a file named exactly `.env` in your project root directory and paste your token inside:
   ```env
   WANDB_API_KEY=your_private_copied_api_key_here
   ```

In [3]:
# Load hidden environment variable tokens safely
load_dotenv()
WANDB_API_KEY = os.getenv("WANDB_API_KEY")

if not WANDB_API_KEY:
    raise ValueError("CRITICAL SECURITY ERROR: 'WANDB_API_KEY' not located in your local .env file!")

# Authenticate the remote experiment tracker programmatically
wandb.login(key=WANDB_API_KEY)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/itzikdermer/.netrc
wandb: Currently logged in as: itzik-dermer (itzik-dermer-technion-israel-institute-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Step 3: Recording Baseline Failure

Before optimizing any weights, let's observe and record how a default pretrained generic model performs on our specific downstream task. 

The cell below hooks into your camera stream, processes frames, and uses native OpenCV video encoders to output an 8-second file directly to your workspace.  
**You need to run this code on your local machine, with a working camera (or hook a USB camera to your PC)**  
**Position your hand clearly in front of the lens.**  


In [4]:
def record_inference_run(weights_path, output_filename):
    """Runs live camera tracking and automatically records an 8-second sample video."""
    if os.path.exists(weights_path):
        print(f"Loading custom optimized model weights from: {weights_path}")
        model = YOLO(weights_path)
    else:
        print(f"Custom weights absent at '{weights_path}'. Loading generic base pretrained model...")
        model = YOLO("yolo26n-pose.pt")
        
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("CRITICAL ERROR: Cannot interface with webcam hardware index.")
        return
        
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    os.makedirs(os.path.dirname(output_filename), exist_ok=True)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_filename, fourcc, 30, (frame_width, frame_height))
    
    print(f"VIDEO RECORDER STARTED -> Target Path: {output_filename}")
    print("Recording will automatically close after 8 seconds. Press 'q' to abort early...")
    
    start_time = time.time()
    while (time.time() - start_time) < 8.0:
        ret, frame = cap.read()
        if not ret: break
        
        # Map OpenCV's native BGR layout to the network's trained RGB profile
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # Explicitly scale the input canvas size to match spatial boundaries
        results = model.predict(rgb_frame, imgsz=640, device=device, verbose=False, conf=0.25)
        
        annotated_frame = results[0].plot()
        out.write(annotated_frame)
        cv2.imshow("YOLOv26 Live Stream Video Recorder", annotated_frame)
        
        if cv2.waitKey(1) & 0xFF == ord('q'): break
        
    cap.release()
    out.release()
    cv2.destroyAllWindows()
    print(f"RECORDING COMPLETE: File saved successfully to {output_filename}\n")

# Execute recording for the failed baseline video
record_inference_run(weights_path="./runs/pose/train/weights/best.pt", output_filename="./runs/pose/predict/failed_attempt.mp4")

Custom weights absent at './runs/pose/train/weights/best.pt'. Loading generic base pretrained model...
VIDEO RECORDER STARTED -> Target Path: ./runs/pose/predict/failed_attempt.mp4
Recording will automatically close after 8 seconds. Press 'q' to abort early...
RECORDING COMPLETE: File saved successfully to ./runs/pose/predict/failed_attempt.mp4



In [5]:
print(ROOT_OUTPUT_DIR)

/Users/itzikdermer/Documents/Python project/Idan Tobis - Deep Learning/HW /Project 3/runs/pose


## Step 4: The Optimization Cycle (Baseline vs. Advanced Custom Engineering)

You will start with a baseline training phase of **50 epochs** using standard parameters. 

Because hand keypoint regression is a highly sensitive task, you will notice that standard linear learning decay schedules and factory default optimizers perform poorly on fine-grained joint details. Once you evaluate the baseline, you are encouraged to disable `RUN_BASELINE` to switch to the Phase 2 advanced custom pipeline (AdamW optimization, Cosine Annealing, explicit webcam blur augmentation simulation, and amplified pose regression loss weights) to build a superior model.

In [7]:
# =====================================================================
# LAB RUN MODE SELECTION
# =====================================================================
# Set to True for Phase 1 Baseline. Set to False to unlock Phase 2 Advanced Tuning.
RUN_BASELINE = False 

# ROOT_OUTPUT_DIR = ROOT_OUTPUT_DIR
##"./runs/pose" 

if RUN_BASELINE:
    print("Running Phase 1: 50-Epoch Baseline Pipeline (Factory Defaults)")
    model = YOLO("yolo26n-pose.pt")
    model.train(
        data="hand-keypoints.yaml",
        epochs=5,                  
        imgsz=640,
        batch=16,
        device=device,
        deterministic=True,
        save=True,
        project=str(ROOT_OUTPUT_DIR),
        name="train"
    )
else:
    print("Running Phase 2: Advanced Custom Optimization Sequence")
    model = YOLO("yolo26n-pose.pt")
    model.train(
        data="hand-keypoints.yaml",
        epochs=10,                 
        imgsz=640,
        batch=16,
        device=device,
        deterministic=True,
        save=True,
        patience=20,
        
        # --- Advanced Optimization Levers ---
        optimizer="AdamW",          # Adaptive gradient scaling
        lr0=0.002,                  # Initial step size tuned for AdamW
        cos_lr=True,                # Smooth Cosine Annealing decay profile
        warmup_epochs=4.0,          # Extended gradient stabilization phase
        
        # --- Loss Coordinate Regularization ---
        pose=18.0,                  # Amplify penalty for misplaced finger joints
        box=5.0,                    # De-emphasize bounding box bounds slightly
        
        # --- Webcam Simulation Augmentations ---
        # blur=0.15,                  # Regularize against rapid hand movements
        scale=0.6,                  # Regularize against variable camera distances
        
        project=str(ROOT_OUTPUT_DIR),
        name="train"
    )

Running Phase 2: Advanced Custom Optimization Sequence
New https://pypi.org/project/ultralytics/8.4.90 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.84 🚀 Python-3.12.13 torch-2.12.1 MPS (Apple M4)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=5.0, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=hand-keypoints.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.002, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-pose.pt, momentum=0.937, 

## Step 5: Verification of Fine-Tuned Tracking

Now that training has completed and generated your custom optimized weights matrix (`./runs/pose/train/weights/best.pt`), run the recording function again. 

### Special Execution Routing for Cloud Users (Colab/Kaggle/Lightning AI)

If you utilized a remote cloud GPU to accelerate your training loop, your browser container cannot interface with your local physical webcam hardware. You must use a hybrid deployment workflow:

#### 1. Download Your Weights Matrix

Once your cloud training run hits completion, navigate to the file explorer sidebar in your cloud environment (Colab or Kaggle). Go to the output directory:
`runs/pose/train/weights/`

Download the **`best.pt`** binary file directly to your local computer.

#### 2. Position the Weights Legally

Move the downloaded `best.pt` file into your local project directory structure, placing it exactly under:
`./runs/pose/train/weights/best.pt`

#### 3. Run Local Hardware Inference

Open your local system terminal, ensure your `uv` environment is activated, and run the detection script locally to open your webcam and record your submission artifacts:

```bash
uv run python detect.py

```


In [8]:
TARGET_CUSTOM_WEIGHTS = "/Users/itzikdermer/Documents/Python project/Idan Tobis - Deep Learning/HW /Project 3/runs/pose/train/weights/best.pt"
## "./runs/pose/train/weights/best.pt"

if not os.path.exists(TARGET_CUSTOM_WEIGHTS):
    print(f"CRITICAL ERROR: Custom fine-tuned weights missing at '{TARGET_CUSTOM_WEIGHTS}'")
    print("Ensure your training loop ran completely and successfully saved its outputs.")
else:
    print("Target weights verified. Launching successful tracking video output recording...")
    record_inference_run(weights_path=TARGET_CUSTOM_WEIGHTS, output_filename="./runs/pose/predict/successful_attempt.mp4")

Target weights verified. Launching successful tracking video output recording...
Loading custom optimized model weights from: /Users/itzikdermer/Documents/Python project/Idan Tobis - Deep Learning/HW /Project 3/runs/pose/train/weights/best.pt
VIDEO RECORDER STARTED -> Target Path: ./runs/pose/predict/successful_attempt.mp4
Recording will automatically close after 8 seconds. Press 'q' to abort early...
RECORDING COMPLETE: File saved successfully to ./runs/pose/predict/successful_attempt.mp4



## Step 6: Git Architecture & Production Repository Deployment

To submit your assignment, you must host your entire working project structure in a public GitHub repository. Follow these commands step-by-step in your local system terminal to initialize, stage, configure, and push your repository.

### 1. Initialize Git and Stage Project Artifacts
Open your system terminal inside your project directory and configure your repository workspace:
```bash
# Initialize a local git ledger tracking snapshot
git init

# Create an explicit ignore rule to prevent bloated binaries or keys from leaking
echo ".env" >> .gitignore
echo ".venv/" >> .gitignore
echo "datasets/" >> .gitignore
echo "*.cache" >> .gitignore

# Stage all configuration profiles, scripts, and video artifacts
git add pyproject.toml hand_pose_lab.ipynb .gitignore
git add runs/pose/predict/failed_attempt.mp4 runs/pose/predict/successful_attempt.mp4
```

### 2. Lock Your Commit
```bash
git commit -m "feat: complete training loop, advanced optimizations, and automated recording sequences"
```

### 3. Provision a Remote GitHub Repository
1. Open your web browser and go to [GitHub](https://github.com).
2. Click the **`+`** icon in the top-right corner and select **New repository**.
3. Set your Repository Name to `hand-pose-estimation`.
4. Ensure visibility is toggled to **Public** (Private submissions cannot be graded).
5. Leave "Add README", "Add .gitignore", and "Choose a license" **UNCHECKED** (since we created them locally).
6. Click **Create repository**.

### 4. Link and Push to Your Remote Repository
Copy the remote commands printed on your GitHub setup screen and run them in your terminal (replace `your-username` with your actual GitHub account name):
```bash
# Define your default branch target naming
git branch -M master

# Route your local repository to your remote cloud space
git remote add origin [https://github.com/your-username/hand-pose-estimation.git](https://github.com/your-username/hand-pose-estimation.git)

# Push your tracked code and video assets to the cloud
git push -u origin master
```

## Step 7: Final Analytical Report Blueprint

Create a file named exactly **`README.md`** in your project root folder, complete the report template below with your tracked training statistics, run `git add README.md`, `git commit -m "docs: finalize lab report"`, and `git push` to upload it. 

Paste the link to your public GitHub repository into Moodle to complete your submission.

```markdown
# Hand Pose Estimation Lab Analysis

### 1. Hardware Profiling & Resource Metrics
* Allocated Target Computing Device: [e.g., Local NVIDIA RTX 4090 / Cloud Kaggle T4 Node]
* Configured Execution Batch Size: [e.g., batch=16 / batch=64]
* Absolute Processing Duration Spent Per Epoch: [e.g., 2 minutes 55 seconds]

### 2. Performance Tracking Metrics Ledger (Best Validated Checkpoint)
* Overall Training Budget Epochs Completed: [e.g., Epoch 68/100]
* Box Loss (box_loss): ___________
* Pose Loss (pose_loss): ___________
* Class Loss (cls_loss): ___________
* Tracking Precision Score (Pose mAP50): ___________
* Rigorous Generalization Bound Score (Pose mAP50-95): ___________

### 3. Optimization and Loss Landscape Analysis
*Paste or embed an image snapshot of your custom Train Loss vs. Validation Loss curves from your Weights & Biases cloud dashboard here.*

* **Critical Evaluation Reflection**: Identify the precise epoch index where your validation loss curve hit its minimum inflection point before stabilizing or rebounding. Explain whether your model ran to your full allocated epoch limit or tripped your early stopping patience threshold. Discuss how your custom optimization changes (e.g., adaptive optimizers, cosine learning curves, and loss weightings) influenced real-world joint tracking accuracy compared to the baseline run.
```